# Task 2 — SQL for Data Extraction
**ApexPlanet Data Analytics Internship**  
**Dataset:** E-commerce Sales (Cleaned)  
**Timeline:** 7 Days (Day 7–13)

---

## 📋 What We Cover
| Days | Topic |
|------|-------|
| Day 7–8 | SQL Fundamentals |
| Day 9–10 | Advanced SQL (CTEs, Window Functions, Views) |
| Day 11–13 | Python + SQL Integration (10 Business Questions) |

---

## 📦 Step 1: Import Libraries

In [ ]:
from IPython.display import display
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ All libraries imported successfully!')

---
## 📂 Step 2: Load Cleaned Data into SQLite Database

In [ ]:
# Load cleaned dataset from Task 1
df = pd.read_csv('../data/data_cleaned.csv')

# Create SQLite database connection
conn = sqlite3.connect('../data/ecommerce.db')

# Load dataframe into SQLite as table named 'orders'
df.to_sql('orders', conn, if_exists='replace', index=False)

print('✅ SQLite database created: data/ecommerce.db')
print(f'✅ Table "orders" loaded with {len(df):,} rows and {len(df.columns)} columns')
print(f'\nColumns: {list(df.columns)}')

In [ ]:
# Helper function — use this every time to run SQL queries
def run_query(query):
    """Run a SQL query and return result as a DataFrame"""
    return pd.read_sql_query(query, conn)

print('✅ Helper function ready! Use run_query("your SQL here") to run queries.')

---
# 📅 DAY 7–8: SQL FUNDAMENTALS

### 🟢 Query 1 — SELECT & LIMIT
View the first few rows of the table (like `df.head()`)

In [ ]:
run_query("""
    SELECT *
    FROM orders
    LIMIT 5
""")

### 🟢 Query 2 — SELECT specific columns
Pick only the columns you need

In [ ]:
run_query("""
    SELECT InvoiceNo, Description, Quantity, UnitPrice, TotalPrice, Country
    FROM orders
    LIMIT 10
""")

### 🟢 Query 3 — WHERE (Filter rows)
Get only orders from the United Kingdom

In [ ]:
run_query("""
    SELECT InvoiceNo, Description, Quantity, UnitPrice, Country
    FROM orders
    WHERE Country = 'United Kingdom'
    LIMIT 10
""")

### 🟢 Query 4 — WHERE with multiple conditions
Orders from UK with TotalPrice greater than 50

In [ ]:
run_query("""
    SELECT InvoiceNo, Description, Quantity, TotalPrice, Country
    FROM orders
    WHERE Country = 'United Kingdom'
      AND TotalPrice > 50
    ORDER BY TotalPrice DESC
    LIMIT 10
""")

### 🟢 Query 5 — ORDER BY (Sort results)
Most expensive products first

In [ ]:
run_query("""
    SELECT Description, UnitPrice
    FROM orders
    ORDER BY UnitPrice DESC
    LIMIT 10
""")

### 🟢 Query 6 — GROUP BY + Aggregate Functions
Count total orders per country

In [ ]:
run_query("""
    SELECT 
        Country,
        COUNT(*)            AS TotalOrders,
        SUM(TotalPrice)     AS TotalRevenue,
        AVG(TotalPrice)     AS AvgOrderValue,
        COUNT(DISTINCT CustomerID) AS UniqueCustomers
    FROM orders
    GROUP BY Country
    ORDER BY TotalRevenue DESC
    LIMIT 10
""")

### 🟢 Query 7 — HAVING (Filter after GROUP BY)
Only show countries with revenue above £10,000

In [ ]:
run_query("""
    SELECT 
        Country,
        ROUND(SUM(TotalPrice), 2) AS TotalRevenue
    FROM orders
    GROUP BY Country
    HAVING TotalRevenue > 10000
    ORDER BY TotalRevenue DESC
""")

### 🟢 Query 8 — LIKE (Pattern Matching)
Find all products with 'CANDLE' in the name

In [ ]:
run_query("""
    SELECT DISTINCT Description, UnitPrice
    FROM orders
    WHERE Description LIKE '%CANDLE%'
    ORDER BY UnitPrice DESC
    LIMIT 10
""")

---
# 📅 DAY 9–10: ADVANCED SQL

### 🔵 Query 9 — Subquery
Find customers who spent more than the average customer spend

In [ ]:
run_query("""
    SELECT 
        CustomerID,
        ROUND(SUM(TotalPrice), 2) AS TotalSpend
    FROM orders
    GROUP BY CustomerID
    HAVING TotalSpend > (
        SELECT AVG(CustomerSpend)
        FROM (
            SELECT SUM(TotalPrice) AS CustomerSpend
            FROM orders
            GROUP BY CustomerID
        )
    )
    ORDER BY TotalSpend DESC
    LIMIT 10
""")

### 🔵 Query 10 — CTE (WITH clause)
Monthly revenue using a CTE — cleaner and more readable than subqueries

In [ ]:
run_query("""
    WITH MonthlyRevenue AS (
        SELECT 
            Year,
            Month,
            ROUND(SUM(TotalPrice), 2) AS Revenue,
            COUNT(DISTINCT CustomerID) AS UniqueCustomers,
            COUNT(DISTINCT InvoiceNo)  AS TotalInvoices
        FROM orders
        GROUP BY Year, Month
    )
    SELECT *
    FROM MonthlyRevenue
    ORDER BY Year, Month
""")

### 🔵 Query 11 — Window Function: ROW_NUMBER
Rank products by revenue within each country

In [ ]:
run_query("""
    WITH ProductRevenue AS (
        SELECT 
            Country,
            Description,
            ROUND(SUM(TotalPrice), 2) AS Revenue
        FROM orders
        GROUP BY Country, Description
    )
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY Country 
            ORDER BY Revenue DESC
        ) AS RankInCountry
    FROM ProductRevenue
    WHERE Country IN ('United Kingdom', 'Germany', 'France')
    QUALIFY RankInCountry <= 3
""")

> ⚠️ Note: If you get an error on the QUALIFY clause (not supported in older SQLite), run the alternative below:

In [ ]:
# Alternative without QUALIFY — works in all SQLite versions
run_query("""
    WITH ProductRevenue AS (
        SELECT 
            Country,
            Description,
            ROUND(SUM(TotalPrice), 2) AS Revenue
        FROM orders
        GROUP BY Country, Description
    ),
    Ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY Country 
                ORDER BY Revenue DESC
            ) AS RankInCountry
        FROM ProductRevenue
        WHERE Country IN ('United Kingdom', 'Germany', 'France')
    )
    SELECT * FROM Ranked
    WHERE RankInCountry <= 3
    ORDER BY Country, RankInCountry
""")

### 🔵 Query 12 — Window Function: RANK
Rank customers by total spending

In [ ]:
run_query("""
    WITH CustomerSpend AS (
        SELECT 
            CustomerID,
            ROUND(SUM(TotalPrice), 2) AS TotalSpend,
            COUNT(DISTINCT InvoiceNo)  AS TotalOrders
        FROM orders
        GROUP BY CustomerID
    )
    SELECT *,
        RANK() OVER (ORDER BY TotalSpend DESC) AS SpendRank
    FROM CustomerSpend
    LIMIT 15
""")

### 🔵 Query 13 — Window Function: LAG (Month-over-Month Growth)
Compare each month's revenue to the previous month

In [ ]:
run_query("""
    WITH MonthlyRevenue AS (
        SELECT 
            Year,
            Month,
            ROUND(SUM(TotalPrice), 2) AS Revenue
        FROM orders
        GROUP BY Year, Month
    )
    SELECT 
        Year,
        Month,
        Revenue,
        LAG(Revenue) OVER (ORDER BY Year, Month) AS PrevMonthRevenue,
        ROUND(
            (Revenue - LAG(Revenue) OVER (ORDER BY Year, Month)) 
            / LAG(Revenue) OVER (ORDER BY Year, Month) * 100, 2
        ) AS GrowthPct
    FROM MonthlyRevenue
    ORDER BY Year, Month
""")

### 🔵 Query 14 — Create a VIEW
Views are saved queries you can reuse anytime like a virtual table

In [ ]:
# Create a reusable view for customer summary
conn.execute("""
    DROP VIEW IF EXISTS customer_summary
""")

conn.execute("""
    CREATE VIEW customer_summary AS
    SELECT 
        CustomerID,
        Country,
        COUNT(DISTINCT InvoiceNo)  AS TotalOrders,
        ROUND(SUM(TotalPrice), 2)  AS TotalSpend,
        ROUND(AVG(TotalPrice), 2)  AS AvgOrderValue,
        MIN(InvoiceDate)           AS FirstPurchase,
        MAX(InvoiceDate)           AS LastPurchase
    FROM orders
    GROUP BY CustomerID, Country
""")

print('✅ View "customer_summary" created!')

# Query the view
run_query("""
    SELECT * FROM customer_summary
    ORDER BY TotalSpend DESC
    LIMIT 10
""")

---
# 📅 DAY 11–13: PYTHON + SQL INTEGRATION
## 10 Business Questions Answered with SQL

### Connect via SQLAlchemy (Professional way)

In [ ]:
# Connect using SQLAlchemy engine
engine = create_engine('sqlite:///../data/ecommerce.db')

def query(sql):
    """Run SQL using SQLAlchemy engine"""
    with engine.connect() as connection:
        return pd.read_sql(text(sql), connection)

print('✅ SQLAlchemy engine connected!')

---
### ❓ Business Question 1 — What are the Top 5 Products by Revenue?

In [ ]:
q1 = query("""
    SELECT 
        Description,
        ROUND(SUM(TotalPrice), 2) AS TotalRevenue,
        SUM(Quantity)             AS TotalUnitsSold
    FROM orders
    GROUP BY Description
    ORDER BY TotalRevenue DESC
    LIMIT 5
""")

print('💡 Q1: Top 5 Products by Revenue')
display(q1)

# Visualize
plt.figure(figsize=(12, 5))
sns.barplot(data=q1, x='TotalRevenue', y='Description', palette='Blues_r')
plt.title('Top 5 Products by Revenue', fontsize=14)
plt.xlabel('Total Revenue (£)')
plt.ylabel('Product')
plt.tight_layout()
plt.savefig('../reports/sql_q1_top_products.png', dpi=150)
plt.show()

### ❓ Business Question 2 — What is the Monthly Sales Trend?

In [ ]:
q2 = query("""
    SELECT 
        Year,
        Month,
        ROUND(SUM(TotalPrice), 2)      AS MonthlyRevenue,
        COUNT(DISTINCT InvoiceNo)       AS TotalOrders,
        COUNT(DISTINCT CustomerID)      AS UniqueCustomers
    FROM orders
    GROUP BY Year, Month
    ORDER BY Year, Month
""")

q2['Period'] = q2['Year'].astype(str) + '-' + q2['Month'].astype(str).str.zfill(2)

print('💡 Q2: Monthly Sales Trend')
display(q2)

plt.figure(figsize=(13, 5))
plt.plot(q2['Period'], q2['MonthlyRevenue'], marker='o', color='steelblue', linewidth=2)
plt.fill_between(range(len(q2)), q2['MonthlyRevenue'], alpha=0.15, color='steelblue')
plt.title('Monthly Sales Revenue Trend (SQL)', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.xticks(range(len(q2)), q2['Period'], rotation=45)
plt.tight_layout()
plt.savefig('../reports/sql_q2_monthly_trend.png', dpi=150)
plt.show()

### ❓ Business Question 3 — Who are the Top 10 Customers by Spend?

In [ ]:
q3 = query("""
    SELECT 
        CustomerID,
        Country,
        ROUND(SUM(TotalPrice), 2)  AS TotalSpend,
        COUNT(DISTINCT InvoiceNo)  AS TotalOrders,
        ROUND(AVG(TotalPrice), 2)  AS AvgOrderValue
    FROM orders
    GROUP BY CustomerID
    ORDER BY TotalSpend DESC
    LIMIT 10
""")

print('💡 Q3: Top 10 Customers by Total Spend')
display(q3)

### ❓ Business Question 4 — Customer Segmentation by Spend Level

In [ ]:
q4 = query("""
    WITH CustomerSpend AS (
        SELECT 
            CustomerID,
            SUM(TotalPrice) AS TotalSpend
        FROM orders
        GROUP BY CustomerID
    )
    SELECT 
        CASE 
            WHEN TotalSpend >= 5000 THEN 'High Value (≥£5000)'
            WHEN TotalSpend >= 1000 THEN 'Mid Value (£1000–£4999)'
            WHEN TotalSpend >= 200  THEN 'Low Value (£200–£999)'
            ELSE 'Occasional (<£200)'
        END AS CustomerSegment,
        COUNT(*)                       AS CustomerCount,
        ROUND(SUM(TotalSpend), 2)      AS SegmentRevenue,
        ROUND(AVG(TotalSpend), 2)      AS AvgSpendPerCustomer
    FROM CustomerSpend
    GROUP BY CustomerSegment
    ORDER BY SegmentRevenue DESC
""")

print('💡 Q4: Customer Segmentation by Spend Level')
display(q4)

# Pie chart
plt.figure(figsize=(8, 6))
plt.pie(q4['CustomerCount'], labels=q4['CustomerSegment'],
        autopct='%1.1f%%', startangle=140,
        colors=['#2196F3','#4CAF50','#FF9800','#F44336'])
plt.title('Customer Segmentation by Spend', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/sql_q4_customer_segments.png', dpi=150)
plt.show()

### ❓ Business Question 5 — Which Countries Generate the Most Revenue?

In [ ]:
q5 = query("""
    SELECT 
        Country,
        ROUND(SUM(TotalPrice), 2)      AS TotalRevenue,
        COUNT(DISTINCT CustomerID)      AS UniqueCustomers,
        COUNT(DISTINCT InvoiceNo)       AS TotalOrders,
        ROUND(AVG(TotalPrice), 2)       AS AvgOrderValue
    FROM orders
    GROUP BY Country
    ORDER BY TotalRevenue DESC
    LIMIT 10
""")

print('💡 Q5: Top 10 Countries by Revenue')
display(q5)

plt.figure(figsize=(12, 5))
sns.barplot(data=q5, x='Country', y='TotalRevenue', palette='Greens_r')
plt.title('Top 10 Countries by Total Revenue', fontsize=14)
plt.xticks(rotation=45)
plt.ylabel('Revenue (£)')
plt.tight_layout()
plt.savefig('../reports/sql_q5_country_revenue.png', dpi=150)
plt.show()

### ❓ Business Question 6 — What is the Best Day of the Week for Sales?

In [ ]:
q6 = query("""
    SELECT 
        Day,
        COUNT(DISTINCT InvoiceNo)      AS TotalOrders,
        ROUND(SUM(TotalPrice), 2)      AS TotalRevenue,
        ROUND(AVG(TotalPrice), 2)      AS AvgOrderValue
    FROM orders
    GROUP BY Day
    ORDER BY TotalRevenue DESC
""")

print('💡 Q6: Sales by Day of Month')
display(q6)

### ❓ Business Question 7 — What is the Average Order Value per Country?

In [ ]:
q7 = query("""
    SELECT 
        Country,
        ROUND(AVG(OrderTotal), 2)   AS AvgOrderValue,
        COUNT(*)                    AS TotalOrders
    FROM (
        SELECT 
            Country,
            InvoiceNo,
            SUM(TotalPrice) AS OrderTotal
        FROM orders
        GROUP BY Country, InvoiceNo
    )
    GROUP BY Country
    HAVING TotalOrders >= 10
    ORDER BY AvgOrderValue DESC
    LIMIT 10
""")

print('💡 Q7: Average Order Value by Country (min 10 orders)')
display(q7)

### ❓ Business Question 8 — Which Products Have Never Been Reordered? (Single Invoice)

In [ ]:
q8 = query("""
    SELECT 
        Description,
        COUNT(DISTINCT InvoiceNo)  AS TimesOrdered,
        SUM(Quantity)              AS TotalQtySold,
        ROUND(SUM(TotalPrice), 2)  AS TotalRevenue
    FROM orders
    GROUP BY Description
    HAVING TimesOrdered = 1
    ORDER BY TotalRevenue DESC
    LIMIT 10
""")

print('💡 Q8: Products Ordered Only Once (No Repeat Orders)')
display(q8)

### ❓ Business Question 9 — Month-over-Month Revenue Growth Rate

In [ ]:
q9 = query("""
    WITH MonthlyRev AS (
        SELECT 
            Year, Month,
            ROUND(SUM(TotalPrice), 2) AS Revenue
        FROM orders
        GROUP BY Year, Month
    )
    SELECT 
        Year, Month, Revenue,
        LAG(Revenue) OVER (ORDER BY Year, Month) AS PrevRevenue,
        ROUND(
            (Revenue - LAG(Revenue) OVER (ORDER BY Year, Month))
            / LAG(Revenue) OVER (ORDER BY Year, Month) * 100, 2
        ) AS GrowthPct
    FROM MonthlyRev
    ORDER BY Year, Month
""")

print('💡 Q9: Month-over-Month Revenue Growth %')
display(q9)

# Plot growth %
q9_clean = q9.dropna(subset=['GrowthPct'])
q9_clean['Period'] = q9_clean['Year'].astype(str) + '-' + q9_clean['Month'].astype(str).str.zfill(2)

colors = ['#4CAF50' if x >= 0 else '#F44336' for x in q9_clean['GrowthPct']]
plt.figure(figsize=(13, 5))
plt.bar(q9_clean['Period'], q9_clean['GrowthPct'], color=colors)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Month-over-Month Revenue Growth (%)', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Growth %')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/sql_q9_mom_growth.png', dpi=150)
plt.show()

### ❓ Business Question 10 — High Value vs Low Value Product Revenue Share

In [ ]:
q10 = query("""
    WITH ProductRevenue AS (
        SELECT 
            Description,
            UnitPrice,
            ROUND(SUM(TotalPrice), 2) AS TotalRevenue
        FROM orders
        GROUP BY Description, UnitPrice
    )
    SELECT 
        CASE 
            WHEN UnitPrice >= 10 THEN 'Premium (≥£10)'
            WHEN UnitPrice >= 3  THEN 'Mid-Range (£3–£9.99)'
            ELSE 'Budget (<£3)'
        END AS PriceCategory,
        COUNT(*)                        AS ProductCount,
        ROUND(SUM(TotalRevenue), 2)     AS TotalRevenue,
        ROUND(AVG(TotalRevenue), 2)     AS AvgRevPerProduct
    FROM ProductRevenue
    GROUP BY PriceCategory
    ORDER BY TotalRevenue DESC
""")

print('💡 Q10: Revenue Share by Product Price Category')
display(q10)

# Donut chart
fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    q10['TotalRevenue'],
    labels=q10['PriceCategory'],
    autopct='%1.1f%%',
    startangle=90,
    colors=['#1565C0', '#42A5F5', '#90CAF9'],
    wedgeprops=dict(width=0.5)
)
ax.set_title('Revenue Share by Product Price Category', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/sql_q10_price_category.png', dpi=150)
plt.show()

---
## 🗃️ Reusable Database Utility Script
Save commonly used queries as Python functions

In [ ]:
# ── Database Utility Functions ──

def get_top_products(n=5):
    """Return top N products by revenue"""
    return query(f"""
        SELECT Description, ROUND(SUM(TotalPrice),2) AS Revenue
        FROM orders GROUP BY Description
        ORDER BY Revenue DESC LIMIT {n}
    """)

def get_country_summary():
    """Return revenue summary by country"""
    return query("""
        SELECT Country,
               ROUND(SUM(TotalPrice),2)     AS Revenue,
               COUNT(DISTINCT CustomerID)   AS Customers,
               COUNT(DISTINCT InvoiceNo)    AS Orders
        FROM orders GROUP BY Country ORDER BY Revenue DESC
    """)

def get_monthly_trend():
    """Return month-by-month revenue"""
    return query("""
        SELECT Year, Month, ROUND(SUM(TotalPrice),2) AS Revenue
        FROM orders GROUP BY Year, Month ORDER BY Year, Month
    """)

def get_customer_summary(top_n=10):
    """Return top N customers by spend"""
    return query(f"""
        SELECT CustomerID, Country,
               ROUND(SUM(TotalPrice),2)  AS TotalSpend,
               COUNT(DISTINCT InvoiceNo) AS TotalOrders
        FROM orders GROUP BY CustomerID
        ORDER BY TotalSpend DESC LIMIT {top_n}
    """)

# ── Test the utility functions ──
print('=== TOP 5 PRODUCTS ===')
display(get_top_products(5))

print('\n=== TOP 5 CUSTOMERS ===')
display(get_customer_summary(5))

---
## 💾 Save SQL Queries to a .sql File

In [ ]:
sql_content = """
-- ============================================================
-- ApexPlanet Data Analytics Internship — Task 2
-- SQL Queries for E-commerce Data Extraction
-- ============================================================

-- Q1: Top 5 Products by Revenue
SELECT Description, ROUND(SUM(TotalPrice),2) AS Revenue
FROM orders GROUP BY Description
ORDER BY Revenue DESC LIMIT 5;

-- Q2: Monthly Sales Trend
SELECT Year, Month, ROUND(SUM(TotalPrice),2) AS MonthlyRevenue
FROM orders GROUP BY Year, Month ORDER BY Year, Month;

-- Q3: Top 10 Customers by Spend
SELECT CustomerID, Country, ROUND(SUM(TotalPrice),2) AS TotalSpend,
       COUNT(DISTINCT InvoiceNo) AS TotalOrders
FROM orders GROUP BY CustomerID ORDER BY TotalSpend DESC LIMIT 10;

-- Q4: Customer Segmentation by Spend
WITH CustomerSpend AS (
    SELECT CustomerID, SUM(TotalPrice) AS TotalSpend
    FROM orders GROUP BY CustomerID
)
SELECT
    CASE
        WHEN TotalSpend >= 5000 THEN 'High Value'
        WHEN TotalSpend >= 1000 THEN 'Mid Value'
        WHEN TotalSpend >= 200  THEN 'Low Value'
        ELSE 'Occasional'
    END AS Segment,
    COUNT(*) AS CustomerCount,
    ROUND(SUM(TotalSpend),2) AS SegmentRevenue
FROM CustomerSpend GROUP BY Segment ORDER BY SegmentRevenue DESC;

-- Q5: Top 10 Countries by Revenue
SELECT Country, ROUND(SUM(TotalPrice),2) AS Revenue,
       COUNT(DISTINCT CustomerID) AS Customers
FROM orders GROUP BY Country ORDER BY Revenue DESC LIMIT 10;

-- Q9: Month-over-Month Growth (Window Function)
WITH MonthlyRev AS (
    SELECT Year, Month, ROUND(SUM(TotalPrice),2) AS Revenue
    FROM orders GROUP BY Year, Month
)
SELECT Year, Month, Revenue,
    LAG(Revenue) OVER (ORDER BY Year, Month) AS PrevRevenue,
    ROUND((Revenue - LAG(Revenue) OVER (ORDER BY Year, Month))
        / LAG(Revenue) OVER (ORDER BY Year, Month) * 100, 2) AS GrowthPct
FROM MonthlyRev ORDER BY Year, Month;
"""

with open('../scripts/task2_queries.sql', 'w') as f:
    f.write(sql_content)

print('✅ SQL queries saved to scripts/task2_queries.sql')

In [ ]:
# Close the connection
conn.close()
print('✅ Database connection closed.')

---
## ✅ Task 2 Complete!

**What we accomplished:**
- ✅ Loaded cleaned dataset into SQLite database
- ✅ Practised SQL fundamentals: SELECT, WHERE, ORDER BY, GROUP BY, HAVING, LIKE
- ✅ Used Advanced SQL: Subqueries, CTEs (WITH), Window Functions (ROW_NUMBER, RANK, LAG)
- ✅ Created a reusable VIEW for customer summary
- ✅ Connected Python to SQLite via SQLAlchemy
- ✅ Answered 10 business questions using SQL + visualizations
- ✅ Built reusable Python database utility functions
- ✅ Saved all queries to `scripts/task2_queries.sql`

**Next Steps:**
1. Push this notebook + `.sql` file to GitHub
2. Record a screen recording
3. Post on LinkedIn and submit on the ApexPlanet portal
4. Move to **Task 3 — Data Visualization & Dashboarding** 🚀

---
*ApexPlanet Data Analytics Internship — Task 2 of 5*